## 0. Setup

In [ ]:
import os, sys, json, math, random, pickle

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))  # notebooks/ -> project root
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
TRAINING_DIR = os.path.join(PROJECT_ROOT, 'Training')
if TRAINING_DIR not in sys.path:
    sys.path.insert(0, TRAINING_DIR)  # so `import config` inside Models/*.py and preprocessing/*.py resolves

import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
except ImportError as e:
    raise ImportError("This notebook needs PyTorch installed (pip install torch).") from e

try:
    import pretty_midi
except ImportError as e:
    raise ImportError("This notebook needs pretty_midi installed (pip install pretty_midi).") from e

import config  # Training/config.py

random.seed(0); np.random.seed(0); torch.manual_seed(0)
print("Project root:", PROJECT_ROOT)
print("Torch:", torch.__version__, "| Device available:", 'cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# ---- Portable, demo-scale config overrides (does NOT modify config.py on disk) ----
config.RAW_MIDI_DIR = os.path.join(PROJECT_ROOT, 'data', 'demo_midi')   # synthetic stand-in corpus, built below
config.PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
config.OUTPUTS_DIR = os.path.join(PROJECT_ROOT, 'outputs')

# Smaller / faster settings so the whole notebook runs in a couple of minutes on CPU.
# Swap these back to the original config.py values (SEQ_LEN=128, EPOCHS=10, ...) for a full run on real MAESTRO data.
DEMO_SEQ_LEN = 64
DEMO_HIDDEN_DIM = 64
DEMO_LATENT_DIM = 16
DEMO_EPOCHS = 5
DEMO_BATCH_SIZE = 8

config.SEQ_LEN = DEMO_SEQ_LEN
config.HIDDEN_DIM = DEMO_HIDDEN_DIM
config.LATENT_DIM = DEMO_LATENT_DIM
config.EPOCHS = DEMO_EPOCHS
config.BATCH_SIZE = DEMO_BATCH_SIZE

for d in [config.RAW_MIDI_DIR, config.PROCESSED_DIR, config.OUTPUTS_DIR,
          os.path.join(config.OUTPUTS_DIR, 'plots'),
          os.path.join(config.OUTPUTS_DIR, 'generated_midis'),
          os.path.join(config.OUTPUTS_DIR, 'preprocessed_samples')]:
    os.makedirs(d, exist_ok=True)

print("Demo config ready. Outputs will be written under:", config.OUTPUTS_DIR)

## 1. Build a small demo MIDI corpus

MAESTRO isn't bundled with the submission, so we synthesize a handful of short piano MIDI files per "style" (standing in for the composer-conditioning used on the real dataset). Each style gets a distinct pitch range / rhythm density so the models have something genuinely separable to learn from.

In [ ]:
STYLES = config.TARGET_COMPOSERS  # ['Frédéric Chopin', 'Johann Sebastian Bach', 'Ludwig van Beethoven', 'Franz Liszt', 'Wolfgang Amadeus Mozart']
FILES_PER_STYLE = 4

STYLE_PROFILES = {
    STYLES[0]: dict(pitch_range=(55, 79), note_len=(0.15, 0.45), density=0.85),   # Chopin: lyrical, mid range
    STYLES[1]: dict(pitch_range=(48, 72), note_len=(0.10, 0.25), density=0.95),   # Bach: dense, contrapuntal
    STYLES[2]: dict(pitch_range=(40, 84), note_len=(0.10, 0.50), density=0.75),   # Beethoven: wide range, dynamic
    STYLES[3]: dict(pitch_range=(60, 96), note_len=(0.08, 0.30), density=0.90),   # Liszt: fast, high register
    STYLES[4]: dict(pitch_range=(52, 76), note_len=(0.20, 0.40), density=0.70),   # Mozart: sparser, balanced
}

def synth_midi(path, profile, n_notes=120, seed=0):
    rng = random.Random(seed)
    midi = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)
    t = 0.0
    lo, hi = profile['pitch_range']
    for _ in range(n_notes):
        if rng.random() < profile['density']:
            pitch = rng.randint(lo, hi)
            dur = rng.uniform(*profile['note_len'])
            vel = rng.randint(60, 110)
            piano.notes.append(pretty_midi.Note(velocity=vel, pitch=pitch, start=t, end=t + dur))
        t += rng.uniform(*profile['note_len'])
    midi.instruments.append(piano)
    midi.write(path)

metadata = []
for style in STYLES:
    for i in range(FILES_PER_STYLE):
        fname = f"{style.replace(' ', '_')}_{i+1}.mid"
        synth_midi(os.path.join(config.RAW_MIDI_DIR, fname), STYLE_PROFILES[style], seed=hash((style, i)) % (2**31))
        metadata.append({"canonical_composer": style, "midi_filename": fname})

with open(os.path.join(config.RAW_MIDI_DIR, 'maestro-v1.0.0.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Synthesized {len(metadata)} demo MIDI files across {len(STYLES)} styles at {config.RAW_MIDI_DIR}")

## 2. Preprocessing

### 2a. Piano-roll representation (for Task 1 / Task 2)
Reuses `preprocessing/midi_parser.py` unchanged.

In [ ]:
from preprocessing.midi_parser import midi_to_piano_roll, segment_piano_roll

all_segments, labels = [], []
for entry in metadata:
    midi_path = os.path.join(config.RAW_MIDI_DIR, entry['midi_filename'])
    pr = midi_to_piano_roll(midi_path, fs=config.FS)
    if pr is None:
        continue
    segs = segment_piano_roll(pr, seq_len=config.SEQ_LEN)
    all_segments.extend(segs)
    labels.extend([config.TARGET_COMPOSERS.index(entry['canonical_composer'])] * len(segs))

X = np.array(all_segments, dtype=np.float32)
y = np.array(labels, dtype=np.int64)
print("Piano-roll dataset:", X.shape, "| labels:", y.shape)

np.save(os.path.join(config.PROCESSED_DIR, 'X.npy'), X)
np.save(os.path.join(config.PROCESSED_DIR, 'y.npy'), y)

# Save >=20 individual preprocessed samples (satisfies the "preprocessed graph/data samples" submission item)
sample_dir = os.path.join(config.OUTPUTS_DIR, 'preprocessed_samples')
n_to_save = min(20, len(X))
for i in range(n_to_save):
    np.save(os.path.join(sample_dir, f'sample_{i:03d}.npy'), X[i])
    with open(os.path.join(sample_dir, f'sample_{i:03d}.json'), 'w') as f:
        json.dump({"index": i, "style": config.TARGET_COMPOSERS[int(y[i])], "shape": list(X[i].shape)}, f)
print(f"Saved {n_to_save} preprocessed samples to {sample_dir}")

In [ ]:
plt.figure(figsize=(6, 3))
plt.imshow(X[0].T, aspect='auto', origin='lower', cmap='Greys')
plt.title(f"Example piano-roll segment (style: {config.TARGET_COMPOSERS[y[0]]})")
plt.xlabel('Time step'); plt.ylabel('Pitch (offset from A0)')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'plots', 'example_piano_roll.png'))
plt.show()

### 2b. Event-token representation (for Task 3 / Task 4)
Reuses `preprocessing/tokenizer.py` unchanged.

In [ ]:
from preprocessing.tokenizer import MIDITokenizer

tokenizer = MIDITokenizer()
token_data = []
for entry in metadata:
    midi_path = os.path.join(config.RAW_MIDI_DIR, entry['midi_filename'])
    toks = tokenizer.midi_to_tokens(midi_path)
    if toks:
        token_data.append(toks)

with open(os.path.join(config.PROCESSED_DIR, 'tokens.pkl'), 'wb') as f:
    pickle.dump(token_data, f)

print(f"Tokenized {len(token_data)} files. Vocab size: {tokenizer.vocab_size}")
print("First 20 tokens of file 0:", token_data[0][:20])

## 3. Task 1 — LSTM Autoencoder
Reuses `Models/autoencoder.py` unchanged (only the import path is fixed: `Models.autoencoder`, not `models.autoencoder`).

In [ ]:
from Models.autoencoder import LSTMAutoencoder
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_t = torch.from_numpy(X).float()
ae_loader = DataLoader(TensorDataset(X_t), batch_size=config.BATCH_SIZE, shuffle=True)

ae_model = LSTMAutoencoder(input_dim=config.NUM_PIANO_NOTES, hidden_dim=config.HIDDEN_DIM,
                            latent_dim=config.LATENT_DIM, seq_len=config.SEQ_LEN).to(device)
ae_opt = optim.Adam(ae_model.parameters(), lr=config.LEARNING_RATE)
ae_criterion = nn.MSELoss()

ae_losses = []
ae_model.train()
for epoch in range(config.EPOCHS):
    epoch_loss = 0.0
    for (batch,) in ae_loader:
        batch = batch.to(device)
        ae_opt.zero_grad()
        recon, _ = ae_model(batch)
        loss = ae_criterion(recon, batch)
        loss.backward()
        ae_opt.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(ae_loader)
    ae_losses.append(avg)
    print(f"[AE] Epoch {epoch+1}/{config.EPOCHS} — Loss: {avg:.4f}")

plt.figure(); plt.plot(ae_losses, marker='o')
plt.title('Task 1: LSTM Autoencoder Reconstruction Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'plots', 'task1_loss.png')); plt.show()

torch.save(ae_model.state_dict(), os.path.join(config.OUTPUTS_DIR, 'task1_ae.pth'))

In [ ]:
# Reconstruction example: input vs. reconstructed piano-roll
ae_model.eval()
with torch.no_grad():
    sample = X_t[0:1].to(device)
    recon, z = ae_model(sample)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(sample[0].cpu().numpy().T, aspect='auto', origin='lower', cmap='Greys'); axes[0].set_title('Original')
axes[1].imshow((recon[0].cpu().numpy() > 0.5).T, aspect='auto', origin='lower', cmap='Greys'); axes[1].set_title('Reconstructed')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'plots', 'task1_reconstruction.png')); plt.show()

## 4. Task 2 — Variational Autoencoder (style-conditioned)
Reuses `Models/vae.py` unchanged.

In [ ]:
from Models.vae import VAE

y_t = torch.from_numpy(y).long()
vae_loader = DataLoader(TensorDataset(X_t, y_t), batch_size=config.BATCH_SIZE, shuffle=True)

vae_model = VAE(input_dim=config.NUM_PIANO_NOTES, hidden_dim=config.HIDDEN_DIM, latent_dim=config.LATENT_DIM,
                seq_len=config.SEQ_LEN, num_genres=len(config.TARGET_COMPOSERS)).to(device)
vae_opt = optim.Adam(vae_model.parameters(), lr=config.LEARNING_RATE)

def vae_loss_fn(x_hat, x, mu, logvar, beta=1.0):
    recon_loss = nn.MSELoss()(x_hat, x)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    kl_loss = kl_loss / (x.size(0) * x.size(1) * x.size(2))
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

vae_losses = []
vae_model.train()
for epoch in range(config.EPOCHS):
    epoch_loss = 0.0
    for x_b, g_b in vae_loader:
        x_b, g_b = x_b.to(device), g_b.to(device)
        vae_opt.zero_grad()
        x_hat, mu, logvar = vae_model(x_b, g_b)
        loss, recon_l, kl_l = vae_loss_fn(x_hat, x_b, mu, logvar)
        loss.backward()
        vae_opt.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(vae_loader)
    vae_losses.append(avg)
    print(f"[VAE] Epoch {epoch+1}/{config.EPOCHS} — Loss: {avg:.4f}")

plt.figure(); plt.plot(vae_losses, marker='o')
plt.title('Task 2: VAE Total Loss (Recon + KL)'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'plots', 'task2_loss.png')); plt.show()

torch.save(vae_model.state_dict(), os.path.join(config.OUTPUTS_DIR, 'task2_vae.pth'))

In [ ]:
# t-SNE of the VAE latent means, colored by style -- addresses the "t-SNE" evaluation item
from sklearn.manifold import TSNE

vae_model.eval()
with torch.no_grad():
    mu_all, _ = vae_model.encode(X_t.to(device))
mu_all = mu_all.cpu().numpy()

perplex = max(2, min(30, len(mu_all) // 3))
tsne = TSNE(n_components=2, random_state=0, perplexity=perplex, init='pca')
emb = tsne.fit_transform(mu_all)

plt.figure(figsize=(6, 5))
for idx, style in enumerate(config.TARGET_COMPOSERS):
    mask = (y == idx)
    plt.scatter(emb[mask, 0], emb[mask, 1], label=style, alpha=0.7)
plt.legend(fontsize=7); plt.title('t-SNE of VAE latent space (colored by style)')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'plots', 'task2_tsne.png')); plt.show()

In [ ]:
# Generate one sample per style from the VAE and write to MIDI
def piano_roll_to_midi(piano_roll_bool, out_path, fs=config.FS):
    midi = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)
    full_pr = np.zeros((128, piano_roll_bool.shape[0]))
    full_pr[config.PIANO_RANGE[0]:config.PIANO_RANGE[1], :] = piano_roll_bool.T
    prev = np.zeros(128)
    for t in range(full_pr.shape[1]):
        for n in range(128):
            if full_pr[n, t] > 0 and prev[n] == 0:
                prev[n] = t
            elif full_pr[n, t] == 0 and prev[n] > 0:
                piano.notes.append(pretty_midi.Note(velocity=100, pitch=n, start=prev[n]/fs, end=t/fs))
                prev[n] = 0
    midi.instruments.append(piano)
    midi.write(out_path)
    return out_path

vae_model.eval()
vae_sample_paths = []
with torch.no_grad():
    for g_idx, style in enumerate(config.TARGET_COMPOSERS):
        z = torch.randn(1, config.LATENT_DIM).to(device)
        g = torch.tensor([g_idx]).to(device)
        x_hat = vae_model.decode(z, g).squeeze(0).cpu().numpy()
        pr_bool = (x_hat > 0.5).astype(np.float32)
        out_path = os.path.join(config.OUTPUTS_DIR, 'generated_midis', f"task2_{style.replace(' ', '_')}_sample.mid")
        piano_roll_to_midi(pr_bool, out_path)
        vae_sample_paths.append((style, out_path))

vae_sample_paths

## 5. Task 3 — Transformer generator
Reuses `Models/transformer.py` unchanged.

In [ ]:
from Models.transformer import MusicTransformer
from torch.utils.data import Dataset

class TokenDataset(Dataset):
    def __init__(self, token_lists, seq_len):
        self.samples = []
        for tl in token_lists:
            step = max(1, seq_len // 2)
            for i in range(0, max(1, len(tl) - seq_len), step):
                chunk = tl[i:i+seq_len]
                if len(chunk) == seq_len:
                    self.samples.append(chunk)
        if not self.samples:  # fall back for very short demo sequences
            for tl in token_lists:
                if len(tl) >= 2:
                    pad = tl + [tokenizer.pad_token] * max(0, seq_len - len(tl))
                    self.samples.append(pad[:seq_len])
        self.samples = torch.LongTensor(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx, :-1], self.samples[idx, 1:]

TF_SEQ_LEN = 48  # short, demo-scale context window
tf_dataset = TokenDataset(token_data, seq_len=TF_SEQ_LEN)
tf_loader = DataLoader(tf_dataset, batch_size=min(config.BATCH_SIZE, max(1, len(tf_dataset))), shuffle=True)
print(f"Transformer training sequences: {len(tf_dataset)}")

tf_model = MusicTransformer(vocab_size=tokenizer.vocab_size, d_model=64, nhead=4, num_layers=2, dim_feedforward=128).to(device)
tf_opt = optim.Adam(tf_model.parameters(), lr=config.LEARNING_RATE)
tf_criterion = nn.CrossEntropyLoss()

tf_losses, tf_ppls = [], []
tf_model.train()
for epoch in range(config.EPOCHS):
    epoch_loss = 0.0
    for x_b, y_b in tf_loader:
        x_b = x_b.transpose(0, 1).to(device)   # (SeqLen, Batch)
        y_b = y_b.transpose(0, 1).to(device)
        mask = tf_model.generate_square_subsequent_mask(x_b.size(0)).to(device)
        tf_opt.zero_grad()
        out = tf_model(x_b, tgt_mask=mask)
        loss = tf_criterion(out.reshape(-1, tokenizer.vocab_size), y_b.reshape(-1))
        loss.backward()
        tf_opt.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(tf_loader)
    ppl = math.exp(min(avg, 20))
    tf_losses.append(avg); tf_ppls.append(ppl)
    print(f"[Transformer] Epoch {epoch+1}/{config.EPOCHS} — Loss: {avg:.4f}, Perplexity: {ppl:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(tf_losses, marker='o'); axes[0].set_title('Task 3: Cross-Entropy Loss')
axes[1].plot(tf_ppls, marker='o', color='orange'); axes[1].set_title('Task 3: Perplexity')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'plots', 'task3_loss.png')); plt.show()

torch.save(tf_model.state_dict(), os.path.join(config.OUTPUTS_DIR, 'task3_transformer.pth'))

In [ ]:
@torch.no_grad()
def generate_tokens(model, tokenizer, max_len=80, device=device):
    model.eval()
    tokens = [tokenizer.bos_token]
    for _ in range(max_len):
        inp = torch.LongTensor(tokens).unsqueeze(1).to(device)
        mask = model.generate_square_subsequent_mask(len(tokens)).to(device)
        logits = model(inp, tgt_mask=mask)
        probs = torch.softmax(logits[-1, 0, :], dim=-1)
        next_tok = torch.multinomial(probs, 1).item()
        tokens.append(next_tok)
        if next_tok == tokenizer.eos_token:
            break
    return tokens

tf_tokens = generate_tokens(tf_model, tokenizer)
tf_midi = tokenizer.tokens_to_midi(tf_tokens[1:-1] if tf_tokens[-1] == tokenizer.eos_token else tf_tokens[1:])
task3_path = os.path.join(config.OUTPUTS_DIR, 'generated_midis', 'task3_transformer_sample.mid')
tf_midi.write(task3_path)
print(f"Generated {len(tf_tokens)} tokens -> {task3_path}")

## 6. Task 4 — RLHF tuning
Reuses the reward function from `Training/rlhf.py` unchanged; only the (buggy) `import src.config` / `from models...` lines are swapped for the fixed imports already set up above.

In [ ]:
def reward_function(tokens):
    if not tokens:
        return 0.0
    reward = 0.0
    pitches = [t for t in tokens if 0 <= t <= 127]
    if pitches:
        reward += (len(set(pitches)) / len(pitches)) * 2.0
    shifts = [t for t in tokens if 256 <= t <= 355]
    if tokens:
        shift_ratio = len(shifts) / len(tokens)
        reward += 1.0 if 0.1 < shift_ratio < 0.4 else -1.0
    c_major = [0, 2, 4, 5, 7, 9, 11]
    if pitches:
        in_scale = sum(1 for p in pitches if p % 12 in c_major)
        reward += (in_scale / len(pitches)) * 1.5
    return reward

rlhf_model = MusicTransformer(vocab_size=tokenizer.vocab_size, d_model=64, nhead=4, num_layers=2, dim_feedforward=128).to(device)
rlhf_model.load_state_dict(tf_model.state_dict())  # start from the Task 3 policy
rlhf_opt = optim.Adam(rlhf_model.parameters(), lr=1e-5)

RLHF_ITERS = 10
rewards = []
rlhf_model.train()
for it in range(RLHF_ITERS):
    tokens = [tokenizer.bos_token]
    log_probs = []
    for _ in range(40):
        inp = torch.LongTensor(tokens).unsqueeze(1).to(device)
        mask = rlhf_model.generate_square_subsequent_mask(len(tokens)).to(device)
        logits = rlhf_model(inp, tgt_mask=mask)
        probs = torch.softmax(logits[-1, 0, :], dim=-1)
        dist = torch.distributions.Categorical(probs)
        tok = dist.sample()
        tokens.append(tok.item())
        log_probs.append(dist.log_prob(tok))
        if tok.item() == tokenizer.eos_token:
            break
    reward = reward_function(tokens[1:])
    rewards.append(reward)

    rlhf_opt.zero_grad()
    policy_loss = torch.stack([-lp * reward for lp in log_probs]).sum()
    policy_loss.backward()
    rlhf_opt.step()
    print(f"[RLHF] Iteration {it+1}/{RLHF_ITERS} — Reward: {reward:.4f}")

plt.figure(); plt.plot(rewards, marker='o')
plt.title('Task 4: RLHF Reward per Iteration'); plt.xlabel('Iteration'); plt.ylabel('Reward')
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'plots', 'task4_reward.png')); plt.show()

torch.save(rlhf_model.state_dict(), os.path.join(config.OUTPUTS_DIR, 'task4_rlhf.pth'))

In [ ]:
rlhf_tokens = generate_tokens(rlhf_model, tokenizer)
rlhf_midi = tokenizer.tokens_to_midi(rlhf_tokens[1:-1] if rlhf_tokens[-1] == tokenizer.eos_token else rlhf_tokens[1:])
task4_path = os.path.join(config.OUTPUTS_DIR, 'generated_midis', 'task4_rlhf_sample.mid')
rlhf_midi.write(task4_path)
print(f"RLHF sample -> {task4_path}")

## 7. Evaluation
Reuses `Evaluation/metrics.py` unchanged (`rhythm_diversity_score`, `repetition_ratio`, `pitch_histogram_similarity`) and combines it with the loss/perplexity figures reported for each stage.

In [ ]:
from Evaluation.metrics import evaluate_midi, get_pitch_histogram, pitch_histogram_similarity
import pandas as pd

reference_path = os.path.join(config.RAW_MIDI_DIR, metadata[0]['midi_filename'])

rows = []
rows.append({"Model": "Task 1: AE", "Loss": round(ae_losses[-1], 4), "Perplexity": None,
             **evaluate_midi(os.path.join(config.OUTPUTS_DIR, 'generated_midis', 'task2_' + STYLES[0].replace(' ', '_') + '_sample.mid') if False else task3_path, reference_path)})
# (AE has no direct MIDI export cell above; reuse VAE/Transformer/RLHF samples for the qualitative MIDI metrics,
#  AE's own quality is already captured by its reconstruction loss/plot in Section 3.)
rows[-1] = {"Model": "Task 1: AE", "Loss": round(ae_losses[-1], 4), "Perplexity": None}

vae_style, vae_path = vae_sample_paths[0]
rows.append({"Model": "Task 2: VAE", "Loss": round(vae_losses[-1], 4), "Perplexity": None,
             **evaluate_midi(vae_path, reference_path)})

rows.append({"Model": "Task 3: Transformer", "Loss": None, "Perplexity": round(tf_ppls[-1], 2),
             **evaluate_midi(task3_path, reference_path)})

rows.append({"Model": "Task 4: RLHF", "Loss": None, "Perplexity": None,
             **evaluate_midi(task4_path, reference_path), "Reward": round(rewards[-1], 4)})

results_df = pd.DataFrame(rows)
results_df

In [ ]:
results_df.to_csv(os.path.join(config.OUTPUTS_DIR, 'evaluation_table.csv'), index=False)

plot_df = results_df.set_index('Model')[['rhythm_diversity', 'repetition_ratio']].dropna(how='all')
plot_df.plot(kind='bar', figsize=(7, 4))
plt.title('Evaluation metrics by model'); plt.ylabel('Score'); plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'plots', 'evaluation_comparison.png')); plt.show()

## 8. End-to-end inference example

Given just a **style name**, this cell runs the full trained pipeline (VAE style-conditioned decode → RLHF-tuned Transformer continuation isn't literally chained in the original design, so here we demonstrate the VAE's conditioned generation path end-to-end, which is the model in this project that takes an explicit style input) and returns a finished, evaluated MIDI file. This is the single inference example required for the submission.

In [ ]:
def run_end_to_end(style_name: str):
    assert style_name in config.TARGET_COMPOSERS, f"style must be one of {config.TARGET_COMPOSERS}"
    g_idx = config.TARGET_COMPOSERS.index(style_name)

    vae_model.eval()
    with torch.no_grad():
        z = torch.randn(1, config.LATENT_DIM).to(device)
        g = torch.tensor([g_idx]).to(device)
        x_hat = vae_model.decode(z, g).squeeze(0).cpu().numpy()
    pr_bool = (x_hat > 0.5).astype(np.float32)

    out_path = os.path.join(config.OUTPUTS_DIR, 'generated_midis', f"demo_inference_{style_name.replace(' ', '_')}.mid")
    piano_roll_to_midi(pr_bool, out_path)

    metrics = evaluate_midi(out_path, reference_path)

    plt.figure(figsize=(6, 3))
    plt.imshow(pr_bool.T, aspect='auto', origin='lower', cmap='Greys')
    plt.title(f"End-to-end inference — style: {style_name}")
    plt.xlabel('Time step'); plt.ylabel('Pitch (offset from A0)')
    plt.tight_layout(); plt.show()

    print(f"Saved MIDI to: {out_path}")
    print("Evaluation metrics:", metrics)
    return out_path, metrics

# Example call — change the style to any entry in config.TARGET_COMPOSERS
demo_path, demo_metrics = run_end_to_end(config.TARGET_COMPOSERS[0])

## 9. Summary

| Output | Location |
|---|---|
| Preprocessed piano-roll dataset | `data/processed/X.npy`, `data/processed/y.npy` |
| ≥20 individual preprocessed samples | `outputs/preprocessed_samples/` |
| Tokenized dataset | `data/processed/tokens.pkl` |
| Loss / perplexity / reward / t-SNE plots | `outputs/plots/` |
| Generated MIDI (all 4 tasks + inference example) | `outputs/generated_midis/` |
| Evaluation table | `outputs/evaluation_table.csv` |
| Trained model checkpoints | `outputs/task{1,2,3,4}_*.pth` |

**To run this on the real MAESTRO dataset:** point `config.RAW_MIDI_DIR` at your local `maestro-v1.0.0` folder (containing `maestro-v1.0.0.json` and the `.midi` files), remove the demo-scale overrides in Section 0 (restore `SEQ_LEN=128`, `HIDDEN_DIM=256`, `EPOCHS=10`, etc.), and re-run from Section 1 onward — every downstream cell is unchanged.